In [21]:
import pandas as pd

# Load all 3 datasets
df1 = pd.read_csv('../data/raw/jobs_in_data_2024.csv')
print("Dataset 1 shape:", df1.shape)
print("Columns:", df1.columns.tolist())

Dataset 1 shape: (14199, 12)
Columns: ['work_year', 'experience_level', 'employment_type', 'job_title', 'salary', 'salary_currency', 'salary_in_usd', 'employee_residence', 'work_setting', 'company_location', 'company_size', 'job_category']


In [22]:
df2 = pd.read_csv('../data/raw/jobs_in_data.csv')
print("Dataset 2 shape:", df2.shape)
print("Columns:", df2.columns.tolist())

Dataset 2 shape: (9355, 12)
Columns: ['work_year', 'job_title', 'job_category', 'salary_currency', 'salary', 'salary_in_usd', 'employee_residence', 'experience_level', 'employment_type', 'work_setting', 'company_location', 'company_size']


In [23]:
df3 = pd.read_csv('../data/raw/postings.csv')
print("Dataset 3 shape:", df3.shape)
print("Columns:", df3.columns.tolist())

Dataset 3 shape: (123849, 31)
Columns: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


In [24]:
import re

def is_real_job_title(title):
    title = str(title)
    if re.search(r'\d', title): return False
    if len(title) > 60: return False
    if any(x in title.lower() for x in [
        'part-time', 'full-time', 'remote', 'hybrid', 'onsite',
        'urgent', 'hiring', 'immediately', 'opening', 'opportunity',
        'shift', 'contract', 'temp', 'monday'
    ]): return False
    return True

d3_clean = df3[['title', 'max_salary']].copy()
d3_clean.columns = ['job_title', 'salary_in_usd']
d3_clean = d3_clean[d3_clean['job_title'].apply(is_real_job_title)]
d3_clean = d3_clean.dropna(subset=['salary_in_usd'])
d3_clean = d3_clean[d3_clean['salary_in_usd'] > 15000]
d3_clean = d3_clean[d3_clean['salary_in_usd'] < 500000]
d3_clean['work_year'] = 2024
d3_clean['experience_level'] = 'Mid-level'
d3_clean['company_location'] = 'United States'
print("Clean postings:", d3_clean.shape)

Clean postings: (16213, 5)


In [25]:
def assign_category(title):
    t = str(title).lower()
    if any(x in t for x in ['data scientist', 'machine learning', 'ml engineer', 'ai engineer', 'deep learning', 'nlp']):
        return 'Data Science and AI'
    elif any(x in t for x in ['data engineer', 'etl', 'pipeline', 'databricks', 'spark']):
        return 'Data Engineering'
    elif any(x in t for x in ['data analyst', 'analytics', 'business analyst', 'bi analyst', 'business intelligence']):
        return 'Analytics and BI'
    elif any(x in t for x in ['software engineer', 'software developer', 'backend', 'frontend', 'full stack', 'web developer', 'ios', 'android']):
        return 'Software Engineering'
    elif any(x in t for x in ['devops', 'cloud engineer', 'site reliability', 'platform engineer', 'infrastructure']):
        return 'DevOps and Cloud'
    elif any(x in t for x in ['product manager', 'product owner', 'program manager']):
        return 'Product Management'
    elif any(x in t for x in ['security', 'cybersecurity', 'information security', 'penetration']):
        return 'Cybersecurity'
    elif any(x in t for x in ['doctor', 'physician', 'nurse', 'medical', 'clinical', 'pharmacist', 'therapist', 'health']):
        return 'Healthcare'
    elif any(x in t for x in ['finance', 'financial', 'accountant', 'accounting', 'tax', 'audit', 'banker', 'investment']):
        return 'Finance and Accounting'
    elif any(x in t for x in ['marketing', 'seo', 'content', 'brand', 'growth', 'social media', 'copywriter']):
        return 'Marketing'
    elif any(x in t for x in ['teacher', 'professor', 'lecturer', 'education', 'instructor', 'tutor', 'curriculum']):
        return 'Education'
    elif any(x in t for x in ['lawyer', 'legal', 'attorney', 'counsel', 'paralegal', 'compliance']):
        return 'Legal'
    elif any(x in t for x in ['hr ', 'human resources', 'recruiter', 'talent', 'people ops', 'payroll']):
        return 'Human Resources'
    elif any(x in t for x in ['sales', 'account executive', 'business development', 'account manager']):
        return 'Sales'
    elif any(x in t for x in ['design', 'ux', 'ui ', 'graphic', 'creative', 'art director', 'visual']):
        return 'Design'
    elif any(x in t for x in ['chef', 'cook', 'culinary', 'kitchen', 'restaurant', 'food']):
        return 'Food and Hospitality'
    elif any(x in t for x in ['director', 'chief', 'cto', 'ceo', 'cfo', 'vp ', 'head of', 'president']):
        return 'Executive and Management'
    elif any(x in t for x in ['consultant', 'consulting', 'advisory', 'strategist']):
        return 'Consulting'
    elif any(x in t for x in ['operations', 'supply chain', 'logistics', 'procurement']):
        return 'Operations'
    elif any(x in t for x in ['research', 'scientist', 'researcher', 'lab', 'biology', 'chemistry']):
        return 'Research and Science'
    else:
        return 'Other'

# Apply to all datasets
d1 = df1[['work_year', 'experience_level', 'job_title', 'salary_in_usd', 'company_location']].copy()
d1['job_category'] = d1['job_title'].apply(assign_category)

d2 = df2[['work_year', 'job_title', 'salary_in_usd', 'employee_residence']].copy()
d2.rename(columns={'employee_residence': 'company_location'}, inplace=True)
d2['experience_level'] = 'Mid-level'
d2['job_category'] = d2['job_title'].apply(assign_category)

d3_clean['job_category'] = d3_clean['job_title'].apply(assign_category)

# Merge
# Only use d1 and d2 - clean international datasets
# d3 has US-only salaries which confuses the model

d1 = df1[['work_year', 'experience_level', 'job_title', 'salary_in_usd', 'company_location']].copy()
d1['job_category'] = d1['job_title'].apply(assign_category)

d2 = df2[['work_year', 'job_title', 'salary_in_usd', 'employee_residence']].copy()
d2.rename(columns={'employee_residence': 'company_location'}, inplace=True)
d2['experience_level'] = 'Mid-level'
d2['job_category'] = d2['job_title'].apply(assign_category)

merged = pd.concat([d1, d2], ignore_index=True)
merged = merged[merged['job_category'] != 'Other']
merged.dropna(subset=['salary_in_usd', 'job_title', 'company_location'], inplace=True)

print("Total merged:", merged.shape)
print("\nCategories:")
print(merged['job_category'].value_counts())
merged.to_csv('../data/processed/merged_jobs.csv', index=False)
print("Saved!")

Total merged: (21421, 6)

Categories:
job_category
Data Science and AI         8296
Data Engineering            5362
Analytics and BI            5173
Research and Science        2043
Executive and Management     179
Consulting                   160
Operations                    80
Product Management            47
DevOps and Cloud              33
Design                        28
Software Engineering          20
Name: count, dtype: int64
Saved!
